# RepoCoder Studio — RAG-Augmented Retrain (Add-On to the Corrected Full Run)

**Fixes:** Implementation Report §13.12, "A fine-tuned model can be less
receptive to RAG evidence than the base model."

## What this notebook is

An **add-on** to `RepoCoderStudio_Fast_Corrected_Retrain.ipynb`, not a
replacement for it. It does **not** rebuild the approved corpus, does
**not** re-evaluate the baseline/fine-tuned model, and does **not**
rebuild the Stage 4/5 indexes — all of that is the original notebook's
job, and its already-committed outputs (`approved_corpus.jsonl`, Stage 4
repository indexes, the Stage 5 corpus index) are the source of truth
behind `RepoCoderStudio_Implementation_Report.md`'s numbers. Duplicating
that construction here would risk quietly producing slightly different
artifacts from a second code path.

**Run `RepoCoderStudio_Fast_Corrected_Retrain.ipynb` first if you haven't
already.** This notebook hard-fails with a clear message naming the
missing file if any required artifact isn't there yet — it does not fall
back to rebuilding anything.

## What this notebook adds

The original notebook's LoRA fine-tuning corpus was built exclusively
with the plain `Instruction -> Input -> Response` template — it never
included a `### Retrieved Context` section, so the fine-tuned model never
learned to condition on injected evidence. Manual Gradio testing proved
this directly: given identical retrieved evidence for a LedgerFlow
email-validation query, the untouched pretrained baseline copied the
repository's exact regex verbatim, while the fine-tuned model ignored the
same evidence and invented its own.

This notebook trains a **new**, separately-named adapter
(`RepoCoderStudio_RAGAware_LoRA_v1_2`) on a corrected, grounded training corpus that
additionally includes RAG-formatted examples, then verifies the fix
against that exact failure case plus the existing hidden-policy
benchmark, then launches Gradio against the new adapter.

## Source modules used by the corrected flow

- `src/prompt_builder_rag.py` — `RAGPromptBuilder.build_rag_training_text()`,
  the training-time counterpart to the existing `build_rag_inference_prompt()`.
- `src/task_builder_rag.py` — `RAGAugmentedTaskDatasetBuilder`, wraps the
  existing `TaskDatasetBuilder` unchanged and adds RAG-formatted examples
  for ~35% of train-split T1-T4 rows. Each augmented row uses its own
  validated target-language reference as relevant-by-construction evidence.
- `src/response_safe_training_rag.py` — extends the original notebook's
  response-safe token-budgeting step to also handle RAG-augmented rows,
  which are long enough to risk overflowing `max_seq_length=1024` and
  crashing the completion collator if left unguarded.
- `src/config.py` — makes the adapter identity, learning rate, dataset name
  and inference context budget explicit and reproducible.
- `src/generation_validation.py` — rejects parseable placeholders such as
  a one-letter identifier instead of reporting them as successful code.

## What's required to already exist (from notebook 1)

| Artifact | Path |
|---|---|
| Approved corpus | `outputs/approved_corpus/approved_corpus.jsonl` |
| Stage 4 LedgerFlow index | `outputs/repo_explorer/...` for `ledgerflow` |
| Stage 5 corpus index | `outputs/corpus_index/*` |

## Steps

1. Install, restart, mount, verify prerequisites exist (hard fail if not).
2. Load the approved corpus.
3. Build the RAG-augmented task dataset.
4. Tokenize + response-safe budgeting.
5. Train the new adapter (skips automatically if it already exists).
6. Verify: the exact email-validation case that exposed the gap.
7. Verify: the hidden transfer-policy benchmark (no-regression check).
8. Save the verification report.
9. FastAPI — documented, not launched (Colab has no Docker runtime).
10. **Gradio — launched, last step.**


## Step 1a — Install dependencies, then restart the kernel

Run once; it restarts the kernel itself (numpy ABI requirement) — Colab reconnects automatically, this is expected. **After it restarts, run the next cell** (do not rerun this one). Skip entirely if you're continuing in the same already-set-up runtime.

In [ ]:
# ============================================================
# Step 1a — Install dependencies, then restart
# ============================================================

import sys
import shutil
import subprocess

if shutil.which("javac") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"])

packages = [
    "numpy==1.26.4",
    "huggingface_hub==0.25.2",
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "peft==0.12.0",
    "trl==0.10.1",
    "bitsandbytes>=0.43,<0.46",
    "sentencepiece",
    "protobuf>=3.20.2,<6",
    "sentence-transformers>=3.0,<4",
    "tree-sitter==0.22.3",
    "tree-sitter-python==0.21.0",
    "tree-sitter-java==0.21.0",
    "faiss-cpu>=1.8,<2",
    "networkx",
    "matplotlib",
    "pandas",
    "fastapi==0.112.2",
    "starlette==0.38.6",
    "uvicorn==0.30.6",
    "gradio-client==1.3.0",
    "gradio==4.44.1",
]

print("Installing RepoCoderStudio runtime dependencies...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", *packages]
)

print("Install complete. Restarting the kernel to load the new numpy build...")
print("After the restart, run the next cell (do not rerun this one).")

import os
os.kill(os.getpid(), 9)


Installing RepoCoderStudio runtime dependencies...


## Step 1b — Mount Drive, verify prerequisites (hard fail if missing)

Run this only after Step 1a has restarted the kernel (or first, if you skipped 1a). If any required artifact from notebook 1 is missing, this cell stops immediately naming the exact missing path — it does not attempt to build anything.

In [1]:
# ============================================================
# Step 1b — Mount Drive, configure, verify GPU + prerequisites
# ============================================================

import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/RepoCoderStudio").resolve()
assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"

required_files = [
    PROJECT_ROOT / "src" / "prompt_builder_rag.py",
    PROJECT_ROOT / "src" / "task_builder_rag.py",
    PROJECT_ROOT / "src" / "response_safe_training_rag.py",
    PROJECT_ROOT / "src" / "generation_engine.py",
    PROJECT_ROOT / "src" / "generation_validation.py",
]
missing_new_files = [str(p) for p in required_files if not p.is_file()]
assert not missing_new_files, (
    "Sync these new files to Drive before running this notebook:\n"
    + "\n".join(missing_new_files)
)

# New adapter name -- keeps this run's output completely separate from
# the original RepoCoderStudio_FastCorrected_LoRA_v1_0 adapter/reports.
os.environ["REPOCODER_ADAPTER_NAME"] = "RepoCoderStudio_RAGAware_LoRA_v1_2"
os.environ["REPOCODER_PROMPT_VERSION"] = "rag_prompt_contract_v1.2"
os.environ["REPOCODER_TRAINING_MANIFEST_VERSION"] = "training_manifest_rag_v1.2"
os.environ["REPOCODER_TRAINING_DATASET_FILENAME"] = "task_dataset_rag_grounded.jsonl"
os.environ["REPOCODER_LEARNING_RATE"] = "5e-5"
os.environ["REPOCODER_MAX_CONTEXT_CHARS"] = "2000"
os.environ["REPOCODER_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["REPOCODER_RUN_MODE"] = "demo"
os.environ["REPOCODER_ENABLE_RAG"] = "true"
os.environ["REPOCODER_ALLOW_MOCK_EMBEDDINGS"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import faiss
import gradio as gr

print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), (
    "No GPU is attached. Select Runtime > Change runtime type > T4 GPU, "
    "restart the session, and rerun from Step 1a."
)

from src.config import CONFIG
from src.storage import ProjectStorageManager
from src.repository_catalog import repository_index_dirs

storage = ProjectStorageManager(CONFIG)

# ------------------------------------------------------------
# Hard-fail prerequisite check: these must already exist from
# RepoCoderStudio_Fast_Corrected_Retrain.ipynb. No fallback build path.
# ------------------------------------------------------------

required_artifacts = {
    "Approved corpus": storage.path(storage.approved_corpus_path()),
    "Stage 5 corpus index": storage.corpus_index_dir() / "corpus_index_manifest.json",
}
ledgerflow_parsed_dir, ledgerflow_embedding_dir = repository_index_dirs("ledgerflow", config=CONFIG)
required_artifacts["Stage 4 LedgerFlow index"] = ledgerflow_embedding_dir / "index_manifest.json"

missing_prereqs = {
    label: str(path) for label, path in required_artifacts.items() if not path.exists()
}
assert not missing_prereqs, (
    "Missing prerequisite artifacts -- run RepoCoderStudio_Fast_Corrected_Retrain.ipynb "
    "first (through its Combined Stage, Stage 4, and Stage 5 corpus-index cells):\n"
    + "\n".join(f"  {label}: {path}" for label, path in missing_prereqs.items())
)

print()
print("All prerequisites found.")
print("Adapter name       :", CONFIG.training.final_adapter_name)
print("Prompt contract    :", CONFIG.experiment.prompt_version)
print("Training manifest :", CONFIG.experiment.training_manifest_version)
print("Training dataset  :", CONFIG.training.task_dataset_filename)
assert CONFIG.training.final_adapter_name == "RepoCoderStudio_RAGAware_LoRA_v1_2"
assert CONFIG.training.task_dataset_filename == "task_dataset_rag_grounded.jsonl"
assert CONFIG.training.learning_rate == 5e-5
assert CONFIG.retrieval.max_context_chars == 2000


Mounted at /content/drive
CUDA available : True
GPU            : Tesla T4

All prerequisites found.
Adapter name       : RepoCoderStudio_RAGAware_LoRA_v1_2
Prompt contract    : rag_prompt_contract_v1.2
Training manifest : training_manifest_rag_v1.2
Training dataset  : task_dataset_rag_grounded.jsonl


## Step 2 — Load the approved corpus

Required to already exist (checked in Step 1b). Loaded directly -- no rebuild path in this notebook.

In [2]:
# ============================================================
# Step 2 — Load the approved corpus
# ============================================================

from src.schemas import ApprovedRow

approved_dicts = storage.load_jsonl(storage.approved_corpus_path())
approved_rows = [ApprovedRow(**row) for row in approved_dicts]
print(f"Loaded approved_rows: {len(approved_rows)}")


09:11:50 | INFO     | RepoCoderStudio.Main | Loaded JSONL: /content/drive/MyDrive/RepoCoderStudio/outputs/approved_corpus/approved_corpus.jsonl (537 rows)
Loaded approved_rows: 537


## Step 3 — Build the RAG-augmented task dataset

Builds the same plain task examples the original run produced (unchanged `TaskDatasetBuilder`), then adds RAG-formatted examples for a sampled subset of train-split T1-T4 rows. Each augmented row uses validated target-language reference code from that same training row. This is extractive-grounding supervision: the context always determines the answer. Validation/test rows and LedgerFlow are excluded from training evidence; the saved Stage 4/5 indexes are used later for unseen verification and serving.

In [3]:
# ============================================================
# Step 3 — Build the RAG-augmented task dataset
# ============================================================

from src.task_builder_rag import RAGAugmentedTaskDatasetBuilder

rag_task_builder = RAGAugmentedTaskDatasetBuilder(
    config=CONFIG, augment_ratio=0.35,
    augment_tasks=("T1", "T2", "T3", "T4"), seed=13,
)
task_examples = rag_task_builder.build(approved_rows)

rag_augmented_count = sum(1 for ex in task_examples if ex.metadata.get("rag_augmented"))
print()
print(f"Total task examples    : {len(task_examples)}")
print(f"RAG-augmented examples : {rag_augmented_count}")
print(f"Plain examples         : {len(task_examples) - rag_augmented_count}")

# Fail before spending GPU time if grounding integrity is broken.
rag_examples = [ex for ex in task_examples if ex.metadata.get("rag_augmented")]
assert rag_examples, "No grounded RAG training rows were produced."
assert all(ex.split == "train" for ex in rag_examples), "RAG augmentation leaked outside train."
assert all(ex.task_id in {"T1", "T2", "T3", "T4"} for ex in rag_examples)
assert all(ex.metadata.get("rag_evidence_strategy") == "validated_same_row_target_reference" for ex in rag_examples)
assert all(ex.output_text.strip() in ex.metadata.get("retrieved_context", "") for ex in rag_examples), (
    "A RAG training row has context that does not determine its supervised answer."
)
assert all(len(ex.metadata.get("retrieved_context", "")) <= 2000 for ex in rag_examples)
assert all("ledgerflow" not in ex.metadata.get("retrieved_context", "").lower() for ex in rag_examples)
print("Grounding preflight     : PASS (train-only, aligned, atomic, LedgerFlow-unseen)")



Grounded RAG Task Dataset Builder  [v2.0]

Task Dataset Builder  [v2.3]
09:12:20 | INFO     | RepoCoderStudio.Main | Saved JSONL: /content/drive/MyDrive/RepoCoderStudio/outputs/task_datasets/task_dataset.jsonl (3222 rows)
09:12:21 | INFO     | RepoCoderStudio.Main | Saved CSV: /content/drive/MyDrive/RepoCoderStudio/outputs/reports/task_distribution_v2_3.csv (18 rows)
09:12:21 | INFO     | RepoCoderStudio.Main | Saved CSV: /content/drive/MyDrive/RepoCoderStudio/outputs/reports/task_family_difficulty_v2_3.csv (18 rows)
09:12:22 | INFO     | RepoCoderStudio.Main | Saved CSV: /content/drive/MyDrive/RepoCoderStudio/outputs/reports/task_dataset_contribution_v2_3.csv (12 rows)
09:12:22 | INFO     | RepoCoderStudio.Main | Saved JSON: /content/drive/MyDrive/RepoCoderStudio/outputs/reports/task_dataset_summary_v2_3.json

Task Dataset Summary  [v2.3]
Approved Rows               : 537
Task Examples               : 3222
Expected Max                : 3222
Skipped Empty Tasks         : 0
Prompt Vers

## Step 4 — Tokenize, then apply response-safe budgeting

Tokenizing reuses the unchanged `TokenizerDatasetBuilder`. Budgeting reuses the original notebook's algorithm (preserve the complete response, truncate only the input), extended by `response_safe_training_rag.py` to handle RAG-augmented rows correctly -- necessary because those rows are long enough to risk overflowing `max_seq_length=1024` and cutting into the response, which would otherwise crash the completion collator outright.

In [4]:
# ============================================================
# Step 4 — Tokenize + response-safe budgeting
# ============================================================

from src.tokenizer_builder import TokenizerDatasetBuilder
from src.response_safe_training_rag import apply_response_safe_budgeting

tokenizer_builder = TokenizerDatasetBuilder(CONFIG)
train_dataset, validation_dataset, test_dataset = tokenizer_builder.build(task_examples)

print(f"Train rows (before budgeting) : {len(train_dataset)}")
print(f"Validation rows               : {len(validation_dataset)}")
print(f"Test rows                     : {len(test_dataset)}")

train_dataset, budget_report = apply_response_safe_budgeting(
    train_dataset, task_examples, config=CONFIG,
    report_path="outputs/reports/training_sequence_budget_report_rag_grounded.json",
)

print()
print(f"Train rows (after budgeting)  : {len(train_dataset)}")
print(f"Retention rate                : {budget_report['retention_rate']:.2%}")
print(f"RAG vs plain (retained)       : {budget_report['retained_rag_vs_plain_counts']}")

assert budget_report["retained_rag_vs_plain_counts"].get("rag_augmented", 0) > 0, (
    "Response-safe budgeting removed every grounded RAG row; stop before training."
)
assert budget_report["response_header_preflight_failures"] == 0
assert budget_report["maximum_retained_sequence_tokens"] <= CONFIG.models.max_seq_length
print("Completion-label preflight: PASS")



Tokenizer Dataset Builder  [v2.6]

HF Dataset Summary  [v2.6]
Train Rows                  : 2997
Validation Rows             : 366
Test Rows                   : 426
Train Task Counts           : {'T1': 547, 'T2': 530, 'T3': 574, 'T4': 536, 'T5': 405, 'T6': 405}
Validation Task Counts      : {'T1': 61, 'T2': 61, 'T3': 61, 'T4': 61, 'T5': 61, 'T6': 61}
Test Task Counts            : {'T1': 71, 'T2': 71, 'T3': 71, 'T4': 71, 'T5': 71, 'T6': 71}
Curriculum                  : round_robin_by_task
Train rows (before budgeting) : 2997
Validation rows               : 366
Test rows                     : 426


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Applying response-safe token budgeting (RAG-aware):   0%|          | 0/2997 [00:00<?, ? examples/s]

Keeping completion-safe training rows:   0%|          | 0/2997 [00:00<?, ? examples/s]


Train rows (after budgeting)  : 2877
Retention rate                : 96.00%
RAG vs plain (retained)       : {'plain': 2424, 'rag_augmented': 453}
Completion-label preflight: PASS


## Step 5 — Train the new LoRA adapter (GPU-heavy step)

Uses the isolated `RepoCoderStudio_RAGAware_LoRA_v1_2` path and a conservative learning rate of `5e-5`. It skips training only when both the weights and the saved adapter manifest match the grounded dataset, response-safe budgeted rows, prompt contract and base model. Checkpoint resume is fingerprinted by the dataset and adapter, so an unrelated checkpoint cannot be resumed here.

In [5]:
# ============================================================
# Step 5 — Train or safely reuse the new LoRA adapter
# ============================================================

import gc
import hashlib
import json
from src.artifact_manifest import file_sha256
from src.response_safe_training_rag import RESPONSE_SAFE_BUDGETING_RAG_VERSION

adapter_dir = CONFIG.storage.project_root() / CONFIG.storage.adapters_dir / CONFIG.training.final_adapter_name
adapter_manifest_path = adapter_dir / "trained_model_manifest.json"
rag_dataset_path = storage.path("outputs/task_datasets/task_dataset_rag_grounded.jsonl")

budgeted_prompt_sha256 = hashlib.sha256(
    "\n".join(str(value) for value in train_dataset["prompt_hash"]).encode("utf-8")
).hexdigest()
expected_adapter_manifest = {
    "adapter_name": CONFIG.training.final_adapter_name,
    "base_model": CONFIG.models.student_model_name,
    "prompt_version": CONFIG.experiment.prompt_version,
    "training_manifest_version": CONFIG.experiment.training_manifest_version,
    "task_dataset_filename": CONFIG.training.task_dataset_filename,
    "task_dataset_sha256": file_sha256(rag_dataset_path),
    "response_safe_budgeting_version": RESPONSE_SAFE_BUDGETING_RAG_VERSION,
    "budgeted_prompt_sha256": budgeted_prompt_sha256,
    "train_rows": len(train_dataset),
}

adapter_weights_exist = adapter_dir.is_dir() and (
    (adapter_dir / "adapter_model.safetensors").is_file()
    or (adapter_dir / "adapter_model.bin").is_file()
)
saved_adapter_manifest = None
if adapter_manifest_path.is_file():
    try:
        saved_adapter_manifest = json.loads(adapter_manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        saved_adapter_manifest = None
adapter_is_reusable = adapter_weights_exist and saved_adapter_manifest == expected_adapter_manifest

if adapter_weights_exist and not adapter_is_reusable:
    raise RuntimeError(
        f"Refusing to reuse incompatible or incomplete adapter: {adapter_dir}. "
        "This notebook intentionally uses the corrected v1_2 adapter identity; remove only "
        "that v1_2 folder if it was created by an interrupted attempt, then rerun Step 5."
    )

if adapter_is_reusable:
    print(f"Validated adapter manifest; safely reusing: {adapter_dir}")
else:
    from src.trainer import RepoCoderTrainer

    trainer_wrapper = RepoCoderTrainer(CONFIG)
    trainer_wrapper.load_model_and_tokenizer()
    trained = trainer_wrapper.train(train_dataset, validation_dataset)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    adapter_manifest_path.write_text(
        json.dumps(expected_adapter_manifest, indent=2), encoding="utf-8"
    )
    print()
    print("New validated adapter saved to:", adapter_dir)
    print("Adapter manifest:", adapter_manifest_path)

    for _name in ("trainer_wrapper", "trained"):
        globals().pop(_name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Validated adapter manifest; safely reusing: /content/drive/MyDrive/RepoCoderStudio/outputs/adapters/RepoCoderStudio_RAGAware_LoRA_v1_2


## Step 6 — Verify the fix

Two checks, using the unchanged `load_gradio_runtime` /
`GenerationEngine` / `GenerationOutputValidator` — not reimplemented:

1. The exact LedgerFlow email-validation case that exposed the gap.
2. The flagship hidden transfer-policy benchmark, to confirm no
   regression on the existing headline result.

In [6]:
# ============================================================
# Step 6a — Verify the fix: email-validation regression case
# ============================================================

import src.gradio_runtime as runtime_module

runtime = runtime_module.load_gradio_runtime(
    project_root=PROJECT_ROOT, reuse={}, repository_ids=("ledgerflow",),
)

engine = runtime.generation_engine
ledgerflow_retrieval = runtime.retrieval_engines["ledgerflow"]

query = (
    "Implement this repository's validate_email(value: str) -> bool function. "
    "Preserve the repository's exact regular-expression pattern and strip input "
    "before matching. Return only syntactically valid multi-line Python code, "
    "with imports and the function definition on separate lines."
)
task_id = "T1"
instruction = engine.prompt_builder.build_instruction(task_id)

outcome = ledgerflow_retrieval.resolve(
    query, task_id=task_id, top_k=1, sources=("repo",),
)
print("RAG decision:", outcome.decision.reason,
      "| used:", outcome.used,
      "| top_score:", round(outcome.decision.top_score, 3),
      "| margin:", round(outcome.decision.score_margin, 3))
print()
for source in outcome.sources:
    rank = source.get("rank")
    name = source.get("name")
    file_path = source.get("file_path")
    score = source.get("score")
    print(f"  #{rank} {name} ({file_path}) score={score}")
print()
assert outcome.used and outcome.context.strip(), "Retriever abstained for the email benchmark."
assert len(outcome.context) <= CONFIG.retrieval.max_context_chars
assert "validate_email" in outcome.context, (
    "The accepted context omitted the benchmark's required repository function."
)

arms = {
    "baseline_no_rag": (runtime.baseline_model, runtime.baseline_tokenizer, ""),
    "baseline_with_rag": (runtime.baseline_model, runtime.baseline_tokenizer, outcome.context),
    "finetuned_no_rag": (runtime.finetuned_model, runtime.finetuned_tokenizer, ""),
    "finetuned_with_rag": (runtime.finetuned_model, runtime.finetuned_tokenizer, outcome.context),
}

email_verification = {"query": query, "task_id": task_id, "rag_decision": outcome.decision.reason,
                       "rag_sources": outcome.sources, "outputs": {}}

for arm_name, (model, tokenizer, context) in arms.items():
    raw = engine.generate(model, tokenizer, instruction, query, task_id=task_id, retrieved_context=context)
    checked = runtime.output_validator.validate(task_id, raw)
    code_out = checked.get("normalized_output", raw)
    email_verification["outputs"][arm_name] = {
        "status": checked.get("status"), "reason": checked.get("reason"),
        "valid": checked.get("valid"), "raw_output": raw, "code": code_out,
    }
    print(f"=== {arm_name} ===")
    print(code_out)
    print()

# Automated signal (in addition to eyeballing): does the RAG arm reproduce
# the repository's exact regex character class -- [\w.+-]+@[\w-]+ -- the
# specific detail the fine-tuned model previously ignored? Written without
# backslash escapes to avoid regex/notebook-generation escaping pitfalls.
repo_pattern_fragment = "[" + chr(92) + "w.+-]+@[" + chr(92) + "w-]+"
finetuned_rag_code = email_verification["outputs"]["finetuned_with_rag"]["code"]
reproduced_repo_pattern = repo_pattern_fragment in finetuned_rag_code
print("Fine-tuned/With RAG reproduces the repository's exact regex pattern:",
      reproduced_repo_pattern)
email_verification["finetuned_rag_reproduces_repo_pattern"] = reproduced_repo_pattern
email_verification["success"] = bool(
    email_verification["outputs"]["finetuned_with_rag"]["valid"]
    and reproduced_repo_pattern
    and "def validate_email" in finetuned_rag_code
)
print("Email benchmark success:", email_verification["success"])


Loading pretrained baseline model...
09:14:33 | INFO     | RepoCoderStudio.Main | Loading base model: Qwen/Qwen2.5-Coder-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading saved fine-tuned LoRA adapter from outputs...
09:15:08 | INFO     | RepoCoderStudio.Main | Loading base model: Qwen/Qwen2.5-Coder-0.5B-Instruct
09:15:11 | INFO     | RepoCoderStudio.Main | Loading LoRA adapter from /content/drive/MyDrive/RepoCoderStudio/outputs/adapters/RepoCoderStudio_RAGAware_LoRA_v1_2
Loading saved Stage 4 index: ledgerflow


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

RAG decision: evidence_accepted | used: True | top_score: 0.813 | margin: 0.813

  #1 validate_email (utils/validators.py) score=0.8128

=== baseline_no_rag ===
import re

def validate_email(value: str) -> bool:
    # Regular expression to match email patterns
    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    
    # Check if the value matches the email pattern
    return bool(re.match(email_pattern, value))

09:16:18 | WARNING  | RepoCoderStudio.Main | Code generation ended before an implementation was produced; retrying once with a guarded minimum completion length.
=== baseline_with_rag ===


=== finetuned_no_rag ===
def is_valid_email(email):
    import re
    return re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email) != None;

=== finetuned_with_rag ===
import re

def validate_email(email: str) -> bool:
    """Check whether a string is a syntactically valid email address."""
    pattern = r"^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$"
    return bool(

In [7]:
# ============================================================
# Step 6b — Verify no regression: hidden transfer-policy benchmark
# ============================================================

policy_query = (
    "Implement the Python function transfer_risk_score(amount, "
    "customer_tenure_days, destination_country, trusted_device) according "
    "to this repository's transfer policy. Preserve exact business "
    "thresholds, weights, country rules, caps and edge cases. Return "
    "Python code only."
)
policy_task_id = "T1"
policy_instruction = engine.prompt_builder.build_instruction(policy_task_id)

policy_outcome = ledgerflow_retrieval.resolve(
    policy_query, task_id=policy_task_id, top_k=1, sources=("repo",),
)
print("RAG decision:", policy_outcome.decision.reason, "| used:", policy_outcome.used)
for source in policy_outcome.sources:
    rank = source.get("rank")
    name = source.get("name")
    file_path = source.get("file_path")
    score = source.get("score")
    print(f"  #{rank} {name} ({file_path}) score={score}")
assert policy_outcome.used and policy_outcome.context.strip(), "Retriever abstained for policy benchmark."
assert len(policy_outcome.context) <= CONFIG.retrieval.max_context_chars
assert "transfer_risk_score" in policy_outcome.context, (
    "The accepted context omitted the benchmark's required repository function."
)

policy_raw = engine.generate(
    runtime.finetuned_model, runtime.finetuned_tokenizer,
    policy_instruction, policy_query, task_id=policy_task_id,
    retrieved_context=policy_outcome.context,
)
policy_checked = runtime.output_validator.validate(policy_task_id, policy_raw)
policy_code = policy_checked.get("normalized_output", policy_raw)
print()
print("=== finetuned_with_rag (new adapter) ===")
print(policy_code)

top_source_is_transfer_policy = bool(
    policy_outcome.sources
    and policy_outcome.sources[0].get("file_path") == "policies/transfer_policy.py"
)
print()
print("Top retrieved source is policies/transfer_policy.py:", top_source_is_transfer_policy)

import ast
try:
    policy_tree = ast.parse(policy_code)
    policy_constants = {node.value for node in ast.walk(policy_tree) if isinstance(node, ast.Constant)}
    policy_function_names = {node.name for node in ast.walk(policy_tree) if isinstance(node, ast.FunctionDef)}
except SyntaxError:
    policy_constants, policy_function_names = set(), set()
required_policy_numbers = {250000, 100000, 40, 25, 30, 20, 15, 100}
required_policy_countries = {"IR", "KP", "SY"}
policy_rules_preserved = bool(
    required_policy_numbers.issubset(policy_constants)
    and required_policy_countries.issubset(policy_constants)
    and "transfer_risk_score" in policy_function_names
)
print("Exact policy constants/countries preserved:", policy_rules_preserved)

policy_verification = {
    "query": policy_query,
    "task_id": policy_task_id,
    "rag_decision": policy_outcome.decision.reason,
    "top_source_is_transfer_policy": top_source_is_transfer_policy,
    "finetuned_with_rag_status": policy_checked.get("status"),
    "finetuned_with_rag_reason": policy_checked.get("reason"),
    "finetuned_with_rag_valid": policy_checked.get("valid"),
    "finetuned_with_rag_raw_output": policy_raw,
    "finetuned_with_rag_code": policy_code,
    "exact_policy_rules_preserved": policy_rules_preserved,
    "success": bool(policy_checked.get("valid") and top_source_is_transfer_policy and policy_rules_preserved),
}


RAG decision: evidence_accepted | used: True
  #1 transfer_risk_score (policies/transfer_policy.py) score=0.8836

=== finetuned_with_rag (new adapter) ===
def transfer_risk_score(amount, customer_tenure_days, destination_country, trusted_device):
    """Calculate the repository's 0-100 cross-border transfer risk score."""
    high_risk_countries = {"IR", "KP", "SY"}
    score = 0.0
    if amount >= 250_000:
        score += 40
    elif amount >= 100_000:
        score += 25
    if customer_tenure_days < 30:
        score += 20
    if destination_country.strip().upper() in high_risk_countries:
        score += 30
    if not trusted_device:
        score += 15
    return min(score, 100.0)

Top retrieved source is policies/transfer_policy.py: True
Exact policy constants/countries preserved: True


## Step 7 — Save the verification report

Written to a new grounded-v1.2 path; does not overwrite `outputs/reports/stage5_four_arm_repository_demo.json` or any other existing report. The report records raw and normalized outputs, retrieval evidence, semantic checks and an explicit overall-success flag.

In [8]:
# ============================================================
# Step 7 — Save verification report
# ============================================================

import json
import time

overall_success = bool(email_verification["success"] and policy_verification["success"])
verification_report = {
    "adapter_name": CONFIG.training.final_adapter_name,
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "fix_reference": "RepoCoderStudio_Implementation_Report.md section 13.12",
    "training_dataset_summary": {
        "total_examples": len(task_examples),
        "rag_augmented_examples": rag_augmented_count,
        "evidence_strategy": "validated_same_row_target_reference",
        "leakage_boundary": "train_split_only; LedgerFlow unseen",
        "context_char_budget": CONFIG.retrieval.max_context_chars,
        "learning_rate": CONFIG.training.learning_rate,
    },
    "email_validation_regression_check": email_verification,
    "hidden_policy_no_regression_check": policy_verification,
    "overall_success": overall_success,
}

report_path = storage.path("outputs/reports/rag_grounded_retrain_v1_2_verification.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(verification_report, indent=2), encoding="utf-8")
print("Saved:", report_path)
print("Overall verification success:", overall_success)


Saved: /content/drive/MyDrive/RepoCoderStudio/outputs/reports/rag_grounded_retrain_v1_2_verification.json
Overall verification success: True


## Step 8 — FastAPI app UI (not launched here)

The FastAPI service (`app/main.py`, static UI under `app/static/`) is
built for **Docker/GCP deployment**, not for running directly inside a
Colab kernel — it's meant to be served behind the project's `Dockerfile`
(see `RepoCoderStudio_Implementation_Report.md` §10.2/§15 for the
deployment path). Colab has no Docker runtime, so this notebook
deliberately does not attempt to launch it in-process here.

To exercise it once the new adapter exists on Drive:

```bash
docker build -t repocoderstudio .
docker run -p 8000:8000 \
    -e REPOCODER_ADAPTER_NAME=RepoCoderStudio_RAGAware_LoRA_v1_2 \
    -e REPOCODER_MAX_CONTEXT_CHARS=2000 \
    -v /path/to/RepoCoderStudio/outputs:/app/outputs \
    repocoderstudio
# then open http://localhost:8000
```

Set `REPOCODER_ADAPTER_NAME` to this notebook's adapter name so the
FastAPI service picks up the RAG-aware adapter rather than the original
one; both can coexist in the same `outputs/adapters/` folder since they
have different names.


## Step 9 — Launch Gradio against the new adapter (last step)

Uses the unchanged `gradio_runtime.py` / `gradio_showcase.py` — the new
adapter loads automatically because `REPOCODER_ADAPTER_NAME` is still set
from Step 1b. A new `gradio.live` link is printed on every run of this
cell; always open the freshly printed link in a new tab rather than
reusing an old one.

In [ ]:
# ============================================================
# Step 9 — Launch Gradio against the new adapter
# ============================================================

import src.gradio_runtime as runtime_module
import src.gradio_showcase as showcase_module

runtime_module.patch_gradio_template_compatibility()

for _name in ("demo", "gradio_demo"):
    _previous = globals().get(_name)
    if _previous is not None and hasattr(_previous, "close"):
        try:
            _previous.close()
        except Exception:
            pass
try:
    gr.close_all()
except Exception:
    pass

demo = showcase_module.build_gradio_showcase(**runtime.ui_arguments())
demo.queue(default_concurrency_limit=1, max_size=20)

print()
print("Launching RepoCoderStudio (RAG-aware adapter)...")
print("A new gradio.live link is printed below on EVERY run of this cell.")
print("Always open THAT fresh link in a NEW browser tab.")

demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True,
    prevent_thread_lock=True,
)
